In [5]:
from config import Config
from core.mesh import Mesh
from core.fvm_solver import PackedBedSolver
from core.kinetics import KineticsEngine
from utils.io_manager import IOManager
import numpy as np

def main():
    cfg = Config()
    mesh = Mesh(cfg)
    solver = PackedBedSolver(cfg, mesh)
    kinetics = KineticsEngine(cfg)
    io = IOManager(cfg)  # This will now auto-detect run_1, run_2, etc.

    T = np.full((cfg.NR, cfg.NZ), cfg.T_INIT)
    P = np.full((cfg.NR, cfg.NZ), cfg.P_ATM)
    W = {c: np.full((cfg.NR, cfg.NZ), cfg.W0[c]) for c in cfg.W0}

    steps_per_min = int(60 / cfg.DT) 

    for step in range(int(cfg.TOTAL_MIN * 60 / cfg.DT)):
        curr_sec = step * cfg.DT
        curr_min = curr_sec / 60.0
        T_amb = (cfg.T_INIT + (cfg.HEATING_RATE / 60.0) * curr_sec)
        
        W, S_gas = kinetics.compute_step(W, T, cfg.DT)
        P = solver.solve_pressure(P, T, S_gas)
        T = solver.solve_heat(T, P, T_amb)

        if step % steps_per_min == 0:
            print(f"Step {step} | {curr_min:.1f} min...")
            io.save_iteration_data(T, P, W, curr_min)

    io.finalize()

if __name__ == "__main__":
    main()

Step 0 | 0.0 min...
Step 600 | 1.0 min...
Step 1200 | 2.0 min...
Step 1800 | 3.0 min...
Step 2400 | 4.0 min...
Step 3000 | 5.0 min...
Step 3600 | 6.0 min...
Step 4200 | 7.0 min...
Step 4800 | 8.0 min...
Step 5400 | 9.0 min...

[SUCCESS] Results saved in /results
